# Group 1: feature processing notebook

Этот ноутбук предназначен для исследовательской обработки признаков группы `1`.

Логика ячеек специально повторяет структуру `src/features/group_1/feature_processor.py`,
чтобы позже перенос был почти механическим.

In [3]:
from pathlib import Path
import pandas as pd
import sys

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

# теперь обычный импорт
from src.utils.spec_converter import create_feature_spec_template
from src.utils.io import load_feature_names_from_txt

In [4]:
# Пути относительно папки notebooks/group_1/
DATA_PATH = Path("../../data/raw/MIPT_hackathon_dataset.csv")
FEATURES_PATH = Path("../../data/feature_groups/features_group_1.txt")
OUTPUT_DIR = Path("../../notebook_outputs/group_1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
df = pd.read_csv(DATA_PATH)
feature_names = load_feature_names_from_txt(FEATURES_PATH)
block_df = df[feature_names].copy()

print("Block shape:", block_df.shape)
display(block_df.head())

Block shape: (5383, 20)


,handed_to_delivery_ts,lead_Счет оплачен,lead_utm_campaign,contact_Источник трафика,lead_Статус заказа на сайте,lead_Дата перехода в Сборку,lead_utm_term,lead_utm_sky,lead_HEIGHT,rejected_ts,lead_group,lifecycle_incomplete,lead_Дата перехода Передан в доставку,lead_Проблема,lead_Квалификация лида,lead_Источник,issued_or_pvz_ts,lead_utm_source,lead_created_at,contact_responsible_user_id
0,1.762243e+09,NaN,NaN,Yandex Direct,NaN,1.761944e+09,NaN,NaN,NaN,NaN,NaN,False,NaN,Суставы и позвоночник,А - лид,artraid.info,1.762505e+09,NaN,1761943225,MGR_0007
1,1.762243e+09,NaN,704258743,NaN,NaN,1.761944e+09,dem_otek_clip1_otek,---autotargeting,NaN,NaN,NaN,False,NaN,Суставы и позвоночник,С - лид,NaN,1.762690e+09,ppc_ru_yandex_otek_dem,1761973775,MGR_0025
2,1.762252e+09,NaN,NaN,mail.ru,NaN,1.761944e+09,NaN,NaN,NaN,NaN,NaN,False,NaN,Варикоз,С - лид,База,1.762682e+09,NaN,1747646007,MGR_0025
3,1.762243e+09,NaN,NaN,NaN,NaN,1.761944e+09,NaN,NaN,NaN,1.764142e+09,NaN,False,NaN,Варикоз,С - лид,Органика,1.762527e+09,NaN,1761975991,MGR_0025
4,1.762245e+09,NaN,703798483,NaN,NaN,1.761944e+09,dem_sustav_clip002,---autotargeting,NaN,NaN,NaN,False,NaN,Суставы и позвоночник,А - лид,NaN,1.762574e+09,ppc_ru_yandex_sustav_dem,1761970672,MGR_0001


## Функции обработки признаков

Названия функций совпадают с private-методами из `feature_processor.py`.

In [ ]:
def _add_width_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Ширина" not in block_df.columns:
        return
    series = pd.to_numeric(block_df["lead_Ширина"], errors="coerce")
    result["lead_Ширина"] = series.fillna(series.median())


def _add_linear_height_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Линейная высота (см)" not in block_df.columns:
        return
    series = pd.to_numeric(block_df["lead_Линейная высота (см)"], errors="coerce")
    result["lead_Линейная высота (см)"] = series.fillna(series.median())
    result["lead_Линейная высота (см)__was_missing"] = series.isna().astype(int)


def _add_payment_type_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Вид оплаты" not in block_df.columns:
        return
    series = (
        block_df["lead_Вид оплаты"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
        .str.lower()
    )
    result["lead_Вид оплаты"] = series.replace({"": "unknown"}).astype(str)


def _add_returned_ts_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "returned_ts" not in block_df.columns:
        return
    ts = pd.to_datetime(block_df["returned_ts"], errors="coerce")
    result["returned_ts"] = ts.astype("string")
    result["returned_ts__is_present"] = ts.notna().astype(int)


def _add_delivery_service_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Служба доставки" not in block_df.columns:
        return
    series = (
        block_df["lead_Служба доставки"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
    )
    value_counts = series.value_counts(dropna=False)
    rare_values = value_counts[value_counts < 10].index
    result["lead_Служба доставки"] = series.replace(list(rare_values), "OTHER").astype(str)


def _add_default_feature(block_df: pd.DataFrame, result: pd.DataFrame, column: str) -> None:
    series = block_df[column]
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        result[column] = series.fillna("").astype(str)
    else:
        result[column] = series

In [ ]:
X_block = pd.DataFrame(index=block_df.index)

for column in block_df.columns:
    if column == "lead_Ширина":
        _add_width_feature(block_df, X_block)
    elif column == "lead_Линейная высота (см)":
        _add_linear_height_feature(block_df, X_block)
    elif column == "lead_Вид оплаты":
        _add_payment_type_feature(block_df, X_block)
    elif column == "returned_ts":
        _add_returned_ts_feature(block_df, X_block)
    elif column == "lead_Служба доставки":
        _add_delivery_service_feature(block_df, X_block)
    else:
        _add_default_feature(block_df, X_block, column)

print("Processed shape:", X_block.shape)
display(X_block.head())

In [ ]:
X_block.to_csv(OUTPUT_DIR / "X_block.csv", index=False)
feature_spec = create_feature_spec_template(X_block)
feature_spec.to_csv(OUTPUT_DIR / "feature_spec.csv", index=False)

print("Saved:")
print(OUTPUT_DIR / "X_block.csv")
print(OUTPUT_DIR / "feature_spec.csv")